In [1]:
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession
import sys
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import IntegerType, DoubleType, TimestampType
sys.path.append(os.path.abspath(".."))
if SparkSession.getActiveSession():
    SparkSession.getActiveSession().stop()
from spark.config import get_spark_session, get_gcs_path

load_dotenv("../.env")

hadoop_home = os.path.abspath("../spark/hadoop")
os.environ["HADOOP_HOME"] = hadoop_home
os.environ["PATH"] += os.pathsep + os.path.join(hadoop_home, "bin")

key_path=os.path.abspath(
    os.path.join("..", os.getenv("GOOGLE_APPLICATION_CREDENTIALS"))
)
jar_path=os.path.abspath("../spark/gcs-connector.jar")

spark=(
    SparkSession.builder.appName("CityBike_Dev")
    .master("local[*]")
    .config("spark.jars", jar_path)
    .getOrCreate()
)

hadoop_conf=spark._jsc.hadoopConfiguration()
hadoop_conf.set(
    "fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem"
)
hadoop_conf.set("google.cloud.auth.service.account.enable", "true")
hadoop_conf.set("google.cloud.auth.service.account.json.keyfile", key_path)
print(f"Spark Version: {spark.version}")

Spark Version: 4.1.2


In [2]:
bucket_name = "logi-lake"
trips_path = f"gs://{bucket_name}/bronze/trips/*"

df_trips = (
    spark.read.option("header", "true")
    .option("inferSchema", "true")
    .csv(trips_path)
)

df_trips.printSchema()
df_trips.show(5, truncate=False)

root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- ended_at: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: double (nullable = true)
 |-- start_lng: double (nullable = true)
 |-- end_lat: double (nullable = true)
 |-- end_lng: double (nullable = true)
 |-- member_casual: string (nullable = true)

+----------------+-------------+-----------------------+-----------------------+---------------------------+----------------+-------------------------+--------------+-----------+------------+-----------------+------------------+-------------+
|ride_id         |rideable_type|started_at             |ended_at               |start_station_name         |start_station_id|end_station_name         |end_station_id|start_lat  |star

In [3]:
test_output_path = f"gs://{bucket_name}/test/spark_write_test"

(
    df_trips.limit(100)
    .write.mode("overwrite")
    .parquet(test_output_path)
)

df_test_read = spark.read.parquet(test_output_path)
print(f"Прочитано строк: {df_test_read.count()}")

Прочитано строк: 100


In [3]:
import os
import sys
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import IntegerType, DoubleType, TimestampType
sys.path.append(os.path.abspath(".."))
if SparkSession.getActiveSession():
    SparkSession.getActiveSession().stop()
from spark.config import get_spark_session, get_gcs_path

spark = get_spark_session("dev-clean-stations")
bronze_path = get_gcs_path("bronze/stations/*.csv")
df_raw = spark.read.option("header", "true").csv(bronze_path)

df_raw.printSchema()
df_raw.show(5, truncate=False)

root
 |-- station_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- lat: string (nullable = true)
 |-- lon: string (nullable = true)
 |-- capacity: string (nullable = true)
 |-- updated_at: string (nullable = true)

+------------------------------------+------------------------------+-----------------+------------------+--------+--------------------------------+
|station_id                          |name                          |lat              |lon               |capacity|updated_at                      |
+------------------------------------+------------------------------+-----------------+------------------+--------+--------------------------------+
|1804892772559499512                 |W 25 St & 8 Ave               |40.74649996333341|-73.99732626293645|33      |2026-07-15 09:17:25.670737+00:00|
|66dd1f44-0aca-11e7-82f6-3863bb44ef7c|Greenpoint Ave & Manhattan Ave|40.73026         |-73.95394         |27      |2026-07-17 07:45:37.482776+00:00|
|06439006-11b6-44f0

In [3]:
df_cleaned=(
    df_raw
    .withColumn("lat", F.col("lat").cast(DoubleType()))
    .withColumn("lon", F.col("lon").cast(DoubleType()))
    .withColumn("capacity", F.col("capacity").cast(IntegerType()))
    .withColumn("updated_at", F.col("updated_at").cast(TimestampType()))
    .sort(F.col("updated_at").desc())
    .dropDuplicates(["station_id"])
)

df_cleaned.printSchema()
df_cleaned.show(5, truncate=False)

root
 |-- station_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- capacity: integer (nullable = true)
 |-- updated_at: timestamp (nullable = true)

+------------------------------------+------------------------+-----------------+------------------+--------+--------------------------+
|station_id                          |name                    |lat              |lon               |capacity|updated_at                |
+------------------------------------+------------------------+-----------------+------------------+--------+--------------------------+
|00284700-9d22-42ce-8485-113fed9879c1|28 Ave & 44 St          |40.76408932350688|-73.91065120697021|19      |2026-07-17 10:51:30.429614|
|002e6e9f-ced2-47b0-bc7e-ccf38c3b4b0f|Prospect Ave & E 151 St |40.814232        |-73.903927        |21      |2026-07-17 10:51:30.429614|
|00967b8f-1a4d-4131-a65a-17fa8ca89e28|41 Ave & 67 St          |40.74446    

In [4]:
silver_path = get_gcs_path("silver/stations")
print(f"Запись в: {silver_path}")
df_cleaned.write.mode("overwrite").parquet(silver_path)

Запись в: gs://logi-lake/silver/stations


In [5]:
status_bronze_path = get_gcs_path("bronze/status")
print(f"Чтение status из: {status_bronze_path}")

df_status_raw = spark.read.json(status_bronze_path)

df_status_raw.printSchema()
df_status_raw.show(5, truncate=False)

Чтение status из: gs://logi-lake/bronze/status
root
 |-- data: struct (nullable = true)
 |    |-- stations: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- is_installed: long (nullable = true)
 |    |    |    |-- is_renting: long (nullable = true)
 |    |    |    |-- is_returning: long (nullable = true)
 |    |    |    |-- last_reported: long (nullable = true)
 |    |    |    |-- num_bikes_available: long (nullable = true)
 |    |    |    |-- num_bikes_disabled: long (nullable = true)
 |    |    |    |-- num_docks_available: long (nullable = true)
 |    |    |    |-- num_docks_disabled: long (nullable = true)
 |    |    |    |-- num_ebikes_available: long (nullable = true)
 |    |    |    |-- num_scooters_available: long (nullable = true)
 |    |    |    |-- num_scooters_unavailable: long (nullable = true)
 |    |    |    |-- station_id: string (nullable = true)
 |    |    |    |-- vehicle_types_available: array (nullable = true)
 |    |

In [11]:
df_exploded=(
    df_status_raw
    .select(
        F.col("last_updated"),
        F.explode("data.stations").alias("station")
    )
    .select(
        F.col("station.station_id").alias("station_id"),
        F.col("station.num_bikes_available").cast(IntegerType()).alias("num_bikes_available"),
        F.col("station.num_docks_available").cast(IntegerType()).alias("num_docks_available"),
        F.col("station.is_installed").cast(IntegerType()).alias("is_installed"),
        F.col("station.is_renting").cast(IntegerType()).alias("is_renting"),
        F.col("station.last_reported").cast(IntegerType()).alias("last_reported_raw"),
        F.col("last_updated").alias("last_updated_raw")
    )
)

df_exploded.printSchema()
df_exploded.show(5, truncate=False)

root
 |-- station_id: string (nullable = true)
 |-- num_bikes_available: integer (nullable = true)
 |-- num_docks_available: integer (nullable = true)
 |-- is_installed: integer (nullable = true)
 |-- is_renting: integer (nullable = true)
 |-- last_reported_raw: integer (nullable = true)
 |-- last_updated_raw: long (nullable = true)

+------------------------------------+-------------------+-------------------+------------+----------+-----------------+----------------+
|station_id                          |num_bikes_available|num_docks_available|is_installed|is_renting|last_reported_raw|last_updated_raw|
+------------------------------------+-------------------+-------------------+------------+----------+-----------------+----------------+
|66dd1f44-0aca-11e7-82f6-3863bb44ef7c|0                  |0                  |0           |0         |1782236251       |1784275278      |
|06439006-11b6-44f0-8545-c9d39035f32a|0                  |0                  |0           |0         |86400     

In [12]:
df_cleaned_status = (
    df_exploded
    .withColumn("last_reported", F.to_timestamp(F.col("last_reported_raw")))
    .withColumn("last_updated", F.to_timestamp(F.col("last_updated_raw")))
    .withColumn("dt", F.to_date(F.col("last_updated")))
    .filter(
        (F.col("is_installed") == 1) & 
        (F.col("is_renting") == 1) & 
        (F.col("last_reported_raw") > 86400)
    )
    .select(
        "station_id", "num_bikes_available", "num_docks_available", 
        "last_reported", "last_updated", "dt"
    )
)

df_cleaned_status.printSchema()
df_cleaned_status.show(5, truncate=False)

root
 |-- station_id: string (nullable = true)
 |-- num_bikes_available: integer (nullable = true)
 |-- num_docks_available: integer (nullable = true)
 |-- last_reported: timestamp (nullable = true)
 |-- last_updated: timestamp (nullable = true)
 |-- dt: date (nullable = true)

+------------------------------------+-------------------+-------------------+-------------------+-------------------+----------+
|station_id                          |num_bikes_available|num_docks_available|last_reported      |last_updated       |dt        |
+------------------------------------+-------------------+-------------------+-------------------+-------------------+----------+
|78aabb95-6195-4aa7-ab0d-5748eeb26bd1|5                  |26                 |2026-07-17 10:56:36|2026-07-17 11:01:18|2026-07-17|
|d979131e-0b9f-4aa2-88c5-46d4ff94abc8|14                 |4                  |2026-07-17 10:59:37|2026-07-17 11:01:18|2026-07-17|
|053312f2-e77e-4674-b28b-ad357fbcb4ee|17                 |45           

In [13]:
status_silver_path = get_gcs_path("silver/status")
print(f"Запись status в: {status_silver_path}")

(
    df_cleaned_status
    .write
    .mode("append")
    .partitionBy("dt")
    .parquet(status_silver_path)
)

Запись status в: gs://logi-lake/silver/status


In [14]:
weather_bronze_path = get_gcs_path("bronze/weather")
print(f"Чтение weather из: {weather_bronze_path}")

df_weather_raw = spark.read.json(weather_bronze_path)

df_weather_raw.printSchema()
df_weather_raw.show(5, truncate=False)

Чтение weather из: gs://logi-lake/bronze/weather
root
 |-- current: struct (nullable = true)
 |    |-- interval: long (nullable = true)
 |    |-- precipitation: double (nullable = true)
 |    |-- temperature_2m: double (nullable = true)
 |    |-- time: string (nullable = true)
 |    |-- wind_speed_10m: double (nullable = true)
 |-- current_units: struct (nullable = true)
 |    |-- interval: string (nullable = true)
 |    |-- precipitation: string (nullable = true)
 |    |-- temperature_2m: string (nullable = true)
 |    |-- time: string (nullable = true)
 |    |-- wind_speed_10m: string (nullable = true)
 |-- elevation: double (nullable = true)
 |-- generationtime_ms: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- timezone: string (nullable = true)
 |-- timezone_abbreviation: string (nullable = true)
 |-- utc_offset_seconds: long (nullable = true)

+---------------------------------------+--------------------------------+-

In [16]:
df_cleaned_weather=(
    df_weather_raw
    .select(
        F.col("latitude"),
        F.col("longitude"),
        F.col("current.temperature_2m").alias("temperature"),
        F.col("current.precipitation").alias("precipitation"),
        F.col("current.wind_speed_10m").alias("wind_speed"),
        F.to_timestamp(F.col("current.time")).alias("observation_time")
    )
    .withColumn("dt", F.to_date(F.col("observation_time")))
    .dropDuplicates(["observation_time", "latitude", "longitude"])
)

df_cleaned_weather.printSchema()
df_cleaned_weather.show(5, truncate=False)

root
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- temperature: double (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- wind_speed: double (nullable = true)
 |-- observation_time: timestamp (nullable = true)
 |-- dt: date (nullable = true)

+---------+---------+-----------+-------------+----------+-------------------+----------+
|latitude |longitude|temperature|precipitation|wind_speed|observation_time   |dt        |
+---------+---------+-----------+-------------+----------+-------------------+----------+
|40.729595|-73.94968|20.9       |0.0          |6.9       |2026-07-17 08:15:00|2026-07-17|
+---------+---------+-----------+-------------+----------+-------------------+----------+



In [17]:
weather_silver_path=get_gcs_path("silver/weather")
print(f"Запись weather в: {weather_silver_path}")

(
    df_cleaned_weather
    .write
    .mode("append")
    .partitionBy("dt")
    .parquet(weather_silver_path)
)

Запись weather в: gs://logi-lake/silver/weather


In [2]:
trips_bronze_path=get_gcs_path("bronze/trips")

df_trips_raw=spark.read.option("header", "true").csv(trips_bronze_path)

df_trips_raw.printSchema()
df_trips_raw.show(5, truncate=False)

root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: string (nullable = true)
 |-- ended_at: string (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: string (nullable = true)
 |-- start_lng: string (nullable = true)
 |-- end_lat: string (nullable = true)
 |-- end_lng: string (nullable = true)
 |-- member_casual: string (nullable = true)

+----------------+-------------+-----------------------+-----------------------+----------------------+----------------+--------------------------+--------------+------------------+------------------+-----------------+------------------+-------------+
|ride_id         |rideable_type|started_at             |ended_at               |start_station_name    |start_station_id|end_station_name          |end_station_id|start_lat        

In [3]:
from pyspark.sql.types import DoubleType

df_cleaned_trips = (
    df_trips_raw
    .withColumn("started_at", F.to_timestamp(F.col("started_at")))
    .withColumn("ended_at", F.to_timestamp(F.col("ended_at")))
    .withColumn("duration_sec", (F.col("ended_at").cast("long") - F.col("started_at").cast("long")))
    .withColumn("start_lat", F.col("start_lat").cast(DoubleType()))
    .withColumn("start_lng", F.col("start_lng").cast(DoubleType()))
    .withColumn("end_lat", F.col("end_lat").cast(DoubleType()))
    .withColumn("end_lng", F.col("end_lng").cast(DoubleType()))
    .withColumn("dt", F.to_date(F.col("started_at")))
    .filter(
        F.col("start_station_id").isNotNull() &
        F.col("end_station_id").isNotNull() &
        (F.col("duration_sec") > 0)
    )
    .dropDuplicates(["ride_id"])
)

df_cleaned_trips.printSchema()
df_cleaned_trips.show(5, truncate=False)

root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- ended_at: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: double (nullable = true)
 |-- start_lng: double (nullable = true)
 |-- end_lat: double (nullable = true)
 |-- end_lng: double (nullable = true)
 |-- member_casual: string (nullable = true)
 |-- duration_sec: long (nullable = true)
 |-- dt: date (nullable = true)

+----------------+-------------+-----------------------+-----------------------+------------------------+----------------+----------------------+--------------+-----------------+------------------+------------------+------------------+-------------+------------+----------+
|ride_id         |rideable_type|started_at             |ended_at             

In [4]:
trips_silver_path = get_gcs_path("silver/trips")
(
    df_cleaned_trips
    .write
    .mode("append")
    .partitionBy("dt")
    .parquet(trips_silver_path)
)
print("Готово!")

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "d:\logi-project\venv\Lib\site-packages\py4j\clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Sasha\AppData\Local\Programs\Python\Python311\Lib\socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
ConnectionResetError: [WinError 10054] Удаленный хост принудительно разорвал существующее подключение

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "d:\logi-project\venv\Lib\site-packages\py4j\java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\logi-project\venv\Lib\site-packages\py4j\clientserver.py", line 566, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkE

ConnectionRefusedError: [WinError 10061] Подключение не установлено, т.к. конечный компьютер отверг запрос на подключение

ConnectionRefusedError: [WinError 10061] Подключение не установлено, т.к. конечный компьютер отверг запрос на подключение

In [6]:
(
    df_cleaned_trips.limit(1000)
    .write
    .mode("overwrite")
    .partitionBy("dt")
    .parquet("file:///D:/logi-project/logs/test-local-write")
)

In [3]:
from spark.config import get_spark_session
spark = get_spark_session("dev-clean-trips")
print(spark.range(5).collect())

[Row(id=0), Row(id=1), Row(id=2), Row(id=3), Row(id=4)]


In [7]:
(
    df_cleaned_trips.limit(1000)
    .write
    .mode("overwrite")
    .partitionBy("dt")
    .parquet(get_gcs_path("silver/trips_test"))
)

In [5]:
df_trips_raw.count()

ConnectionRefusedError: [WinError 10061] Подключение не установлено, т.к. конечный компьютер отверг запрос на подключение